# Prompt Optimization with OPRO, Optuna, and DSPy

Suppose you are building a pipeline that classifies student essays as *meets* or *does not meet standard*. You have 100 labeled examples but no rubric — only ground truth. What prompt should you use?

Writing prompts by hand and testing them on a few examples works for a few iterations. But finding the best instruction, and the best supporting examples to include alongside it, requires treating prompt selection as a **search problem.** An instruction and a set of few-shot demonstrations together define a prompt — and there are far too many combinations to explore manually.

This notebook solves that problem two ways. The first approach builds the search machinery from scratch to keep every design decision visible: **OPRO** (Optimization by PROmpting) searches over instructions by using an LLM to propose improvements, guided by a running leaderboard of what has worked so far; **Optuna** then searches over which training examples to include as few-shot demonstrations using Bayesian optimization. The second approach uses **DSPy**, which packages the same two searches into a composable framework and treats the prompt as a program with learnable parameters. Seeing both approaches makes it easier to know when to reach for the framework — and what it is doing under the hood.

## 1. Setup

All LLM calls route through a single `client` object. Flip `BACKEND` to switch between OpenAI and Groq — the rest of the notebook stays the same. We also define two shared helpers used throughout: `build_messages` constructs a chat messages list from an instruction plus optional few-shot examples, and `score_prompt` runs that prompt against a dataset and returns accuracy and a list of failures.

In [ ]:
#| output: false
import json
import random
import warnings
warnings.filterwarnings("ignore")

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from openai import OpenAI
from groq import Groq

# ── Backend toggle ────────────────────────────────────────────────────────────
BACKEND = "openai"   # "openai" | "groq"

if BACKEND == "openai":
    MODEL  = "gpt-4o-mini"
    client = OpenAI()
elif BACKEND == "groq":
    MODEL  = "llama-3.3-70b-versatile"
    client = Groq()

random.seed(42)
np.random.seed(42)


def build_messages(instruction: str, few_shots: list, essay: str) -> list:
    """Build a chat messages list from an instruction, few-shot examples, and essay."""
    messages = [{"role": "system", "content": instruction}]
    for ex in few_shots:
        messages.append({"role": "user",      "content": ex["essay"]})
        messages.append({"role": "assistant", "content": ex["label"]})
    messages.append({"role": "user", "content": essay})
    return messages


def classify(instruction: str, few_shots: list, essay: str) -> str:
    """Classify a single essay. Returns the model's prediction (lowercased)."""
    return client.chat.completions.create(
        model=MODEL,
        messages=build_messages(instruction, few_shots, essay),
        temperature=0,
        max_tokens=10,
    ).choices[0].message.content.strip().lower()


def score_prompt(instruction: str, few_shots: list, dataset: list, subsample: int | None = None):
    """
    Evaluate instruction + few_shots on dataset.
    Returns (accuracy, failures) where each failure is {essay, predicted, correct}.
    """
    eval_set = dataset
    if subsample and subsample < len(dataset):
        eval_set = random.sample(dataset, subsample)

    correct, failures = 0, []
    for item in eval_set:
        pred = classify(instruction, few_shots, item["essay"])
        if pred == item["label"].lower():
            correct += 1
        else:
            failures.append({
                "essay":     item["essay"][:300],
                "predicted": pred,
                "correct":   item["label"],
            })
    return correct / len(eval_set), failures

## 2. Data

We use the **Automated Student Assessment Prize (ASAP)** dataset [@asap2012], which contains student essays each paired with a numeric quality score. It is available on HuggingFace as `qanastek/ASAP-AES`. Each essay has a `domain1_score` and a `maximum_score`. We turn this into a binary label: an essay *meets the standard* if it scores at least 60% of the maximum.

Before committing to that threshold, we examine the score distribution.

In [ ]:
#| output: false
ds  = load_dataset("qanastek/ASAP-AES", trust_remote_code=True)
df  = pd.DataFrame(ds["train"])
df["score_ratio"] = df["domain1_score"] / df["maximum_score"]
print(f"Total essays: {len(df):,}")
df[["domain1_score", "maximum_score", "score_ratio"]].describe().round(2)

The normalized ratio `domain1_score / maximum_score` places all essay sets on a common 0–1 scale regardless of how each set was originally scored. We plot the distribution and mark candidate thresholds to decide where to draw the boundary.

In [ ]:
#| code-fold: true
#| label: fig-score-dist
#| fig-cap: "Distribution of normalized essay scores across the full ASAP dataset. The dashed line at 0.6 sits at a natural trough in the distribution, making it a principled threshold between lower- and higher-quality essays."

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
pal = sns.color_palette("muted")

# Left: full distribution with threshold marked
ax = axes[0]
ax.hist(df["score_ratio"], bins=40, color=pal[0], edgecolor="white", linewidth=0.4, alpha=0.85)
ax.axvline(0.6, color="crimson", linestyle="--", linewidth=1.8, label="threshold = 0.60")
ax.set_xlabel("score / maximum score")
ax.set_ylabel("count")
ax.set_title("Score Ratio Distribution")
ax.grid(linestyle="dotted", alpha=0.5)
ax.legend()
ax.set_xlim(0, 1)
sns.despine(ax=ax)

# Right: class balance at the chosen threshold
ax2 = axes[1]
labels_preview = (df["score_ratio"] >= 0.6).map({True: "meets", False: "does not meet"})
counts = labels_preview.value_counts()
ax2.bar(counts.index, counts.values, color=[pal[2], pal[3]], edgecolor="white", linewidth=0.5)
for i, (label, val) in enumerate(counts.items()):
    ax2.text(i, val + 20, f"{val:,}\n({val/len(df)*100:.1f}%)", ha="center", fontsize=9)
ax2.set_ylabel("count")
ax2.set_title("Class Balance at threshold = 0.60")
ax2.grid(axis="y", linestyle="dotted", alpha=0.5)
ax2.set_ylim(0, counts.max() * 1.18)
sns.despine(ax=ax2)

fig.tight_layout()
plt.show()

The 0.60 threshold sits at a natural trough in the distribution and produces a reasonable class balance. We binarize and draw a stratified sample of 150 essays, split into 50 train / 50 val / 50 test.

In [ ]:
THRESHOLD = 0.60

df["label"] = df.apply(
    lambda r: "meets" if r["domain1_score"] >= r["maximum_score"] * THRESHOLD else "does not meet",
    axis=1,
)

# Stratified sample so class balance is preserved across splits
from sklearn.model_selection import train_test_split

sample = df[["essay", "label"]].sample(n=150, random_state=42)
train_df, temp_df = train_test_split(sample, test_size=100, stratify=sample["label"], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=50,  stratify=temp_df["label"],  random_state=42)

train = train_df.to_dict("records")
val   = val_df.to_dict("records")
test  = test_df.to_dict("records")

def label_dist(split):
    counts = pd.Series([x["label"] for x in split]).value_counts()
    return dict(counts)

for name, split in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {label_dist(split)}")

## 3. Baseline

Before optimizing anything we need a reference point. The simplest possible prompt is a single instruction with no examples and no description of what the standard means. We evaluate it on the validation set and inspect the failures — this tells us what kinds of errors we are starting from.

In [ ]:
BASELINE = (
    "Classify the following student essay as 'meets' or 'does not meet' the standard. "
    "Reply with only one of those two exact phrases."
)

val_true, val_pred = [], []
for item in val:
    pred = classify(BASELINE, [], item["essay"])
    val_true.append(item["label"].lower())
    val_pred.append(pred)

baseline_acc = sum(p == t for p, t in zip(val_pred, val_true)) / len(val)
print(f"Baseline val accuracy: {baseline_acc:.2f}  ({int(baseline_acc * len(val))}/{len(val)})")

In [ ]:
#| code-fold: true
#| label: fig-baseline-cm
#| fig-cap: "Confusion matrix for the baseline zero-shot prompt on the 50-example validation set."
from sklearn.metrics import confusion_matrix, classification_report

LABELS = ["meets", "does not meet"]
cm = confusion_matrix(val_true, val_pred, labels=LABELS)

fig, ax = plt.subplots(figsize=(4.5, 3.5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=LABELS, yticklabels=LABELS,
    ax=ax, linewidths=0.5, linecolor="white",
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Baseline — Confusion Matrix (val)")
fig.tight_layout()
plt.show()

print(classification_report(val_true, val_pred, target_names=LABELS))

Let's look at a few misclassified essays to build intuition about what the baseline gets wrong.

In [ ]:
failures = [
    {"essay": item["essay"], "predicted": pred, "correct": true}
    for item, pred, actual in zip(val, val_pred, val_true)
    if pred != actual
]

for f in failures[:3]:
    print(f"Correct: {f['correct']:20s}  |  Predicted: {f['predicted']}")
    print(f"Essay excerpt: {f['essay'][:280]}")
    print("-" * 72)

## 4. OPRO — Instruction Search

### The idea

OPRO [@opro2023] treats instruction optimization as an iterative search. At each step, a *meta-LLM* sees two things: a leaderboard of instructions tried so far (sorted by their validation accuracy), and a sample of essays that the current best instruction got wrong. From this context it proposes new instruction candidates that might do better. The best candidates are scored on the validation set and added to the leaderboard. The loop repeats.

The key insight is that the meta-LLM can read the failure patterns and propose instructions that address them — effectively doing what a human prompt engineer does, but automatically and at scale.

### The meta-prompt

The meta-prompt sent to the optimizer LLM has three parts:

1. **Task description** — what the classifier must do and what format its output must follow
2. **Leaderboard** — the last 10 instructions tried, sorted low → high by accuracy, so the model can see what has and hasn't worked
3. **Failure examples** — a random sample of essays the current best instruction got wrong, so the model can reason about what to fix

The model is asked to return a JSON array of new instruction strings. No explanation, no markdown — just the array.

In [ ]:
SEED_INSTRUCTIONS = [
    "Classify the following essay as 'meets' or 'does not meet' the standard.",
    (
        "You are a strict evaluator. Does this essay meet the required standard? "
        "Reply only with 'meets' or 'does not meet'."
    ),
    (
        "Read the essay carefully and decide if it meets graduate-level writing standards. "
        "Output only 'meets' or 'does not meet'."
    ),
]


def propose_instructions(leaderboard: list, failures: list, n: int = 3) -> list:
    """
    Ask the LLM to propose n better instructions given the current leaderboard and failures.
    Returns a list of instruction strings.
    """
    sorted_board = sorted(leaderboard, key=lambda x: x[1])
    history_str = "\n".join(
        f'[{score:.2f}] "{instr}"' for instr, score in sorted_board[-10:]
    )
    sample_failures = random.sample(failures, min(5, len(failures)))
    failures_str = "\n".join(
        f"- Essay: \"{f['essay'][:200]}...\" → predicted '{f['predicted']}', correct: '{f['correct']}'"
        for f in sample_failures
    )

    meta_prompt = f"""You are an expert prompt engineer optimizing a binary essay classifier.
The classifier must output exactly 'meets' or 'does not meet' — nothing else.

Instructions tried so far (sorted low → high accuracy):
{history_str}

Essays misclassified by the best instruction:
{failures_str}

Generate {n} new instruction candidates likely to perform better.
Analyze the failure patterns above before proposing. Return ONLY a JSON array of strings."""

    raw = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": meta_prompt}],
        temperature=1.0,
        max_tokens=600,
    ).choices[0].message.content.strip()

    # Strip markdown code fences if the model adds them
    raw = raw.replace("```json", "").replace("```", "").strip()
    return json.loads(raw)

We now run the OPRO loop. To keep costs down, each trial scores the instruction on a **subsample** of 30 validation examples rather than all 50. At the end we re-evaluate the top 3 instructions on the full validation set for an honest ranking.

**Approximate API calls:** `3 seeds × 30 + 8 steps × 3 proposals × 30 + 3 final re-evals × 50 = 990 + 150 = ~1,140 classifier calls` plus `8` meta-LLM calls.

In [ ]:
def run_opro(val, steps=8, proposals_per_step=3, subsample=30):
    """Run the OPRO instruction-search loop. Returns (best_instruction, leaderboard)."""
    print("Phase 1: OPRO — instruction search")
    print("-" * 50)

    leaderboard = []

    # Seed with hand-written starting points
    for instr in SEED_INSTRUCTIONS:
        score, _ = score_prompt(instr, [], val, subsample=subsample)
        leaderboard.append((instr, score))
        print(f"  seed  {score:.2f}  |  {instr[:70]}")

    for step in range(steps):
        best_instr, best_score = max(leaderboard, key=lambda x: x[1])
        _, failures = score_prompt(best_instr, [], val, subsample=subsample)
        print(f"\n  step {step + 1}/{steps}  |  best so far: {best_score:.2f}")

        if not failures:
            print("  perfect score on subsample — stopping early")
            break

        candidates = propose_instructions(leaderboard, failures, n=proposals_per_step)
        for instr in candidates:
            score, _ = score_prompt(instr, [], val, subsample=subsample)
            leaderboard.append((instr, score))
            print(f"    {score:.2f}  |  {instr[:70]}")

    # Re-score top 3 on the full validation set for an honest ranking
    print("\n  Re-evaluating top 3 on full val set...")
    top3 = sorted(leaderboard, key=lambda x: x[1], reverse=True)[:3]
    final_board = []
    for instr, _ in top3:
        score, _ = score_prompt(instr, [], val)
        final_board.append((instr, score))
        print(f"    {score:.2f}  |  {instr[:70]}")

    best_instruction, best_score = max(final_board, key=lambda x: x[1])
    print(f"\n  Best instruction (val acc {best_score:.2f}):")
    print(f'  "{best_instruction}"')
    return best_instruction, leaderboard


best_instruction, opro_leaderboard = run_opro(val, steps=8, proposals_per_step=3, subsample=30)

In [ ]:
#| code-fold: true
#| label: fig-opro-leaderboard
#| fig-cap: "OPRO leaderboard: every instruction tried plotted by its (subsampled) validation accuracy. Seed instructions are shown in orange; LLM-proposed candidates in blue. The dashed line marks the baseline accuracy."

scores = [s for _, s in opro_leaderboard]
n_seeds = len(SEED_INSTRUCTIONS)

pal = sns.color_palette("muted")
fig, ax = plt.subplots(figsize=(9, 3.8))

# Seed instructions
ax.scatter(
    range(n_seeds), scores[:n_seeds],
    color=pal[1], s=60, zorder=3, label="seed",
)
# Proposed candidates
ax.scatter(
    range(n_seeds, len(scores)), scores[n_seeds:],
    color=pal[0], s=45, alpha=0.75, zorder=3, label="proposed",
)
ax.axhline(baseline_acc, color="grey", linestyle="--", linewidth=1.2, label=f"baseline ({baseline_acc:.2f})")

best_idx = int(np.argmax(scores))
ax.scatter([best_idx], [scores[best_idx]], color="crimson", s=90, zorder=4, label=f"best ({scores[best_idx]:.2f})")

ax.set_xlabel("trial index")
ax.set_ylabel("val accuracy (subsampled)")
ax.set_title("OPRO — Instruction Search Leaderboard")
ax.grid(linestyle="dotted", alpha=0.5)
ax.legend()
sns.despine()
fig.tight_layout()
plt.show()

## 5. Optuna — Few-Shot Selection

### Why Bayesian optimization?

The instruction is now fixed. The remaining question is: which training examples should we include in the prompt as demonstrations?

This is a combinatorial search problem. With 50 training examples and up to 4 shots, there are $\binom{50}{4} \approx 230{,}000$ possible 4-shot configurations, plus all smaller subsets. We cannot enumerate them.

Optuna [@akiba2019optuna] solves this with **Tree-structured Parzen Estimation (TPE)**, a Bayesian optimization algorithm. TPE builds a probabilistic model of which configurations tend to score well and uses it to decide where to sample next — spending more trials near promising regions and fewer in regions that have consistently underperformed. This makes it far more efficient than random search at the same trial budget.

We encode the search space as two hyperparameters: `n_shots` (how many demos to include, 0–4) and one categorical index per slot (`shot_0`, `shot_1`, ...) choosing which training example fills that slot.

In [ ]:
def run_optuna(instruction, train, val, n_trials=40, max_shots=4, subsample=30):
    """Search for the best few-shot configuration for a fixed instruction."""
    print("Phase 2: Optuna — few-shot selection")
    print("-" * 50)

    def objective(trial):
        n_shots = trial.suggest_int("n_shots", 0, max_shots)

        # Sample shot indices without replacement
        available = list(range(len(train)))
        chosen = []
        for i in range(n_shots):
            if not available:
                break
            idx = trial.suggest_categorical(f"shot_{i}", available)
            chosen.append(idx)
            available = [x for x in available if x != idx]

        few_shots = [train[i] for i in chosen]
        score, _ = score_prompt(instruction, few_shots, val, subsample=subsample)
        return score

    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    # Reconstruct best few-shots from the best trial's parameters
    best_params = study.best_params
    n_shots = best_params["n_shots"]
    best_few_shots = [train[best_params[f"shot_{i}"]] for i in range(n_shots)]

    print(f"\n  Best val accuracy (subsampled): {study.best_value:.2f}")
    print(f"  n_shots = {n_shots}")
    for i, ex in enumerate(best_few_shots):
        print(f"  [{i}] label={ex['label']:12s}  |  essay={ex['essay'][:80]}...")

    return best_few_shots, study


best_few_shots, optuna_study = run_optuna(
    best_instruction, train, val, n_trials=40, max_shots=4, subsample=30
)

In [ ]:
#| code-fold: true
#| label: fig-optuna-history
#| fig-cap: "Optuna optimization history. Each dot is one trial; the red line tracks the running best. TPE concentrates trials in regions that have performed well, so the best value typically improves sharply early and then plateaus."


fig, ax = plt.subplots(figsize=(9, 3.8))
trial_values = [t.value for t in optuna_study.trials if t.value is not None]
best_so_far  = [max(trial_values[:i+1]) for i in range(len(trial_values))]

pal = sns.color_palette("muted")
ax.scatter(range(len(trial_values)), trial_values, color=pal[0], s=30, alpha=0.6, label="trial")
ax.plot(range(len(best_so_far)), best_so_far, color="crimson", linewidth=1.8, label="best so far")
ax.axhline(baseline_acc, color="grey", linestyle="--", linewidth=1.2, label=f"baseline ({baseline_acc:.2f})")
ax.set_xlabel("trial")
ax.set_ylabel("val accuracy (subsampled)")
ax.set_title("Optuna — Few-Shot Search History")
ax.grid(linestyle="dotted", alpha=0.5)
ax.legend()
sns.despine()
fig.tight_layout()
plt.show()

We now have a fully optimized prompt: the best instruction from OPRO paired with the best few-shot examples from Optuna. Let's record its validation accuracy before moving on.

In [ ]:
opro_optuna_val_acc, _ = score_prompt(best_instruction, best_few_shots, val)
print(f"OPRO + Optuna val accuracy: {opro_optuna_val_acc:.2f}  ({int(opro_optuna_val_acc * len(val))}/{len(val)})")
print(f"\nOptimized instruction:")
print(f'  "{best_instruction}"')
print(f"\nFew-shot demonstrations ({len(best_few_shots)}):")
for i, ex in enumerate(best_few_shots):
    print(f"  [{i}] {ex['label']:20s}  |  {ex['essay'][:80]}...")

## 6. DSPy — from First Principles

### 6.1 The core idea

Most prompt frameworks treat the prompt as a static string you write by hand. **DSPy** [@dspy2023] treats it as a *program with learnable parameters.* The two things you can learn are exactly what we just searched for manually:

1. The **instruction** — the text that tells the model what to do
2. The **demonstrations** — which labeled examples to include as few-shot context

Instead of writing these by hand or searching with an external optimizer, DSPy separates the *specification* (what the task is) from the *prompt* (how to express that task to a specific LLM). You write a `Signature` that declares the task as typed input/output fields. A DSPy **optimizer** then compiles the program — it runs the training data through your module, figures out which examples and instructions work best, and bakes them into the prompt automatically.

The key shift in mindset: you are writing a *program specification*, not a *prompt*.

### 6.2 Building blocks

DSPy has three core abstractions:

**`Signature`** — declares the task as typed I/O. Each field has a name, a type, and a description. A `Signature` is not itself a prompt; it is a specification that DSPy uses to *build* a prompt. This separation means the same `Signature` can be compiled into different prompts for different LLMs or different few-shot budgets.

**`Predict`** — the simplest module. It wraps a `Signature` and calls the LLM once, asking it to fill in the output field(s) directly.

**`ChainOfThought`** — adds a `reasoning` output field before the declared output. The LLM is asked to think step-by-step before giving the final answer. Same `Signature`, different module — this is the composability DSPy is designed around.

In [ ]:
#| output: false
import dspy

# Configure the LM — DSPy uses LiteLLM routing under the hood
if BACKEND == "openai":
    lm = dspy.LM("openai/gpt-4o-mini", temperature=0)
elif BACKEND == "groq":
    lm = dspy.LM("groq/llama-3.3-70b-versatile", temperature=0)

dspy.configure(lm=lm)


# ── Signature ──────────────────────────────────────────────────────────────────
class EssayGrading(dspy.Signature):
    """Classify a student essay as meeting or not meeting the writing standard."""

    essay: str = dspy.InputField(desc="The student essay to evaluate")
    label: str = dspy.OutputField(
        desc="Either 'meets' or 'does not meet' — nothing else"
    )


# ── Module ─────────────────────────────────────────────────────────────────────
cot_module = dspy.ChainOfThought(EssayGrading)

# ── Training data as DSPy Examples ────────────────────────────────────────────
trainset = [
    dspy.Example(essay=x["essay"], label=x["label"]).with_inputs("essay")
    for x in train
]
valset = [
    dspy.Example(essay=x["essay"], label=x["label"]).with_inputs("essay")
    for x in val
]


# ── Metric ─────────────────────────────────────────────────────────────────────
def exact_match(example, pred, trace=None):
    return example.label.lower().strip() == pred.label.lower().strip()

We can see the difference between `Predict` and `ChainOfThought` by running both on the same essay. `Predict` returns only the label; `ChainOfThought` adds a `reasoning` field that shows the model's step-by-step thinking before the final answer.

In [ ]:
sample_essay = val[0]["essay"]

# Predict — direct answer
predict_module = dspy.Predict(EssayGrading)
pred_result = predict_module(essay=sample_essay)
print("Predict output:")
print(f"  label: {pred_result.label}")

# ChainOfThought — reasoning first, then answer
cot_result = cot_module(essay=sample_essay)
print("\nChainOfThought output:")
print(f"  reasoning: {cot_result.reasoning[:200]}...")
print(f"  label: {cot_result.label}")

Before optimizing, we evaluate the uncompiled module zero-shot to get a DSPy baseline. `dspy.Evaluate` runs the module on a dataset and aggregates the metric.

In [ ]:
evaluator = dspy.Evaluate(devset=valset, metric=exact_match, display_progress=False, return_all_scores=False)
dspy_baseline_acc = evaluator(cot_module) / 100.0   # dspy.Evaluate returns 0-100
print(f"DSPy zero-shot val accuracy: {dspy_baseline_acc:.2f}  ({int(dspy_baseline_acc * len(val))}/{len(val)})")

### 6.3 BootstrapFewShot — how it works

BootstrapFewShot is the simplest DSPy optimizer and the one to reach for first. It answers the question: *which training examples should I include as few-shot demonstrations?*

The algorithm has three steps:

1. **Forward pass.** Run each training example through the module with zero-shot prompting. Record the full trace — the input, the model's chain-of-thought reasoning, and its predicted output.

2. **Filter.** Keep only the traces where the model's prediction matched the ground truth label. These are "good" demonstrations: examples where the model's reasoning process led to a correct answer. The intuition is that a demonstration is only worth including if the model can already handle that kind of example — bad demonstrations add noise.

3. **Inject.** At inference time, randomly sample a subset of the collected traces and prepend them to every new prompt. The LLM now sees complete worked examples (input → reasoning → correct output) before it sees the query.

No meta-LLM is needed. The optimizer bootstraps its own demonstrations from the data.

In [ ]:
bfs_optimizer = dspy.BootstrapFewShot(
    metric=exact_match,
    max_bootstrapped_demos=4,   # max demos mined from training traces  # <1>
    max_labeled_demos=4,        # max hand-labeled demos to also consider  # <2>
)
compiled_bfs = bfs_optimizer.compile(cot_module, trainset=trainset)

bfs_val_acc = evaluator(compiled_bfs) / 100.0
print(f"BootstrapFewShot val accuracy: {bfs_val_acc:.2f}  ({int(bfs_val_acc * len(val))}/{len(val)})")

1. Demonstrations mined by running training examples forward and keeping correct traces.
2. A separate budget for directly including labeled examples as-is (no forward pass needed).

We can inspect exactly what the compiled module will send to the LLM — the instruction the Signature produced and the demonstrations BootstrapFewShot selected.

In [ ]:
#| code-fold: true
#| code-summary: "Show compiled prompt"
# Run one example to populate the LM call history, then inspect it
_ = compiled_bfs(essay=val[0]["essay"])
dspy.inspect_history(n=1)

### 6.4 MIPROv2 — joint instruction and demo optimization

BootstrapFewShot only searches the demonstration space — it never changes the instruction. MIPROv2 [@mipro2024] extends the search to cover both at once.

Concretely, MIPROv2 does two things that BootstrapFewShot does not:

- **Instruction proposals.** A meta-LLM is given the task description, sample inputs/outputs, and failed traces, and asked to propose better instruction variants — the same idea as OPRO.
- **Joint optimization.** Candidate (instruction, demo subset) combinations are evaluated using a Bayesian search procedure — the same idea as Optuna.

MIPROv2 is therefore the unification of everything we built in Sections 4 and 5, packaged inside the DSPy framework.

:::{.callout-note}
`auto="light"` runs a small number of instruction proposals and Bayesian trials — enough to demonstrate the method without a large API budget. Use `auto="medium"` or `auto="heavy"` for production runs.

:::

In [ ]:
mipro_optimizer = dspy.MIPROv2(
    metric=exact_match,
    auto="light",          # "light" | "medium" | "heavy"
    verbose=False,
)
compiled_mipro = mipro_optimizer.compile(
    cot_module,
    trainset=trainset,
    valset=valset[:20],    # MIPROv2 uses a valset to guide its Bayesian search
    requires_permission_to_run=False,
)

mipro_val_acc = evaluator(compiled_mipro) / 100.0
print(f"MIPROv2 val accuracy: {mipro_val_acc:.2f}  ({int(mipro_val_acc * len(val))}/{len(val)})")

Inspecting MIPROv2's output shows not just the selected demos but also the optimized instruction — which may differ substantially from the original `Signature` docstring.

In [ ]:
#| code-fold: true
#| code-summary: "Show compiled prompt"
_ = compiled_mipro(essay=val[0]["essay"])
dspy.inspect_history(n=1)

## 7. Comparison & Final Evaluation

All methods were tuned on the validation set. We now run a single honest evaluation on the **held-out test set** for each method to compare them fairly.

In [ ]:
testset_dspy = [
    dspy.Example(essay=x["essay"], label=x["label"]).with_inputs("essay")
    for x in test
]
test_evaluator = dspy.Evaluate(
    devset=testset_dspy, metric=exact_match, display_progress=False, return_all_scores=False
)

# Evaluate all methods on test
baseline_test,     _ = score_prompt(BASELINE,          [],             test)
opro_test,         _ = score_prompt(best_instruction,  [],             test)
opro_optuna_test,  _ = score_prompt(best_instruction,  best_few_shots, test)
bfs_test             = test_evaluator(compiled_bfs)   / 100.0
mipro_test           = test_evaluator(compiled_mipro) / 100.0

results = {
    "Baseline (zero-shot)": {"val": baseline_acc,        "test": baseline_test},
    "OPRO (instruction only)": {"val": None,             "test": opro_test},
    "OPRO + Optuna": {"val": opro_optuna_val_acc,        "test": opro_optuna_test},
    "DSPy BootstrapFewShot": {"val": bfs_val_acc,        "test": bfs_test},
    "DSPy MIPROv2": {"val": mipro_val_acc,               "test": mipro_test},
}

for method, scores in results.items():
    val_str  = f"{scores['val']:.2f}" if scores["val"] is not None else "—"
    test_str = f"{scores['test']:.2f}"
    print(f"{method:30s}  val={val_str:5s}  test={test_str}")

In [ ]:
#| code-fold: true
#| label: fig-comparison
#| fig-cap: "Test-set accuracy for all methods. The dashed line marks the zero-shot baseline."

pal = sns.color_palette("muted", n_colors=len(results))
methods = list(results.keys())
test_scores = [results[m]["test"] for m in methods]

fig, ax = plt.subplots(figsize=(9, 3.8))
bars = ax.barh(methods[::-1], test_scores[::-1], color=pal[::-1], edgecolor="white", height=0.55)
ax.axvline(baseline_test, color="grey", linestyle="--", linewidth=1.3, label=f"baseline ({baseline_test:.2f})")

for bar, score in zip(bars, test_scores[::-1]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{score:.2f}", va="center", fontsize=9)

ax.set_xlabel("test accuracy")
ax.set_title("Method Comparison — Held-Out Test Set")
ax.set_xlim(0, 1.05)
ax.grid(axis="x", linestyle="dotted", alpha=0.5)
ax.legend()
sns.despine()
fig.tight_layout()
plt.show()

### When to use each approach

| Method | Optimizes | Needs meta-LLM | Search strategy | Approx. API calls |
|---|---|---|---|---|
| OPRO | instruction only | ✓ | LLM proposals | ~1,000 |
| Optuna | few-shots only | ✗ | Bayesian (TPE) | ~1,200 |
| OPRO + Optuna | both (sequential) | ✓ | LLM + Bayesian | ~2,200 |
| BootstrapFewShot | few-shots only | ✗ | correctness filter | ~50 |
| MIPROv2 | both (joint) | ✓ | LLM + Bayesian | ~1,500+ |

: {tbl-colwidths="[22,18,15,20,25]"}

**OPRO + Optuna vs. DSPy MIPROv2.** These are conceptually identical — both search over instructions and demonstrations — but with different implementations. OPRO + Optuna is transparent and easily customizable: you control the meta-prompt, the search space encoding, and the subsampling strategy. MIPROv2 abstracts this away and integrates with DSPy's module system, which matters when your pipeline has multiple LLM calls (e.g., an agent that classifies, then summarizes, then decides). For a single-step classifier, either works. For multi-step pipelines, DSPy's composability is a significant advantage.

**BootstrapFewShot** is worth trying first — it uses no meta-LLM, costs almost nothing, and often gets you most of the way there. Upgrade to MIPROv2 if the instruction itself is the bottleneck.

**Reproducibility caveat.** LLM outputs are non-deterministic. Re-running OPRO will produce different leaderboards; re-running MIPROv2 will produce a different compiled instruction. The test accuracy numbers reported here reflect a single run on 50 examples — treat them as indicative rather than definitive. Larger eval sets and multiple runs are needed for reliable comparisons in production.

:::{.callout-caution}
Subsampling the validation set during optimization (here, 30 of 50 examples per trial) introduces variance. The best instruction on a 30-sample subsample is not necessarily the best on the full set. This is a deliberate cost–quality tradeoff: full evaluation on every trial would be 5× more expensive. Always re-evaluate the winner on the full validation set before reporting, as we did at the end of `run_opro`.

:::